# Diagnosing your SBI
Now we've run an SBI fit we need to diagnose that it actually works!

In [ ]:
from pathlib import Path

from mach3sbitools.inference import InferenceHandler
from mach3sbitools.simulator import Simulator
from mach3sbitools.utils import MaCh3Logger

MaCh3Logger("mach3sbitools")

## Directly Comparing Model and SBI Likelihoods
The first and easiest thing we can do is compare the log-likelihood prediction from the SBI and Model likelihoods. We produce 2 plots. One plotting the LLHs against each other in 2D and the other showing the distributions in 1D

In [ ]:
# Let's first load the model. This is the same as the CLI `mach3sbi inference` step
prior = Path("prior/my_prior.pkl")
model = Path("models/my_sbi.ckpt")

physics_config = Path("physics_configs/PhysicsConfig.yaml")


# Now we load the simulator
simulator = Simulator(
    "my_simulator",
    "MySimulator",
    physics_config,
)

# Now we grab the inference
inference_handler = InferenceHandler(prior)
inference_handler.load_posterior(model, posterior_config=None)

In [ ]:
# Now we can compare the LLH
from mach3sbitools.diagnostics import compare_logl

# First step is to do a really rough pass
compare_logl(simulator, inference_handler, 10_000, n_bins=100)

In [ ]:
# Looks okay, let's zoom in on the 2D plot
compare_logl(
    simulator, inference_handler, 100_000, likelihood_range=[-2, 0.5], n_bins=100
)

Comparison of LLH is one thing, but we can go even further with simulation based calibration. The suite of techniques are full described here https://sbi.readthedocs.io/en/stable/how_to_guide/diagnostics.html but we provide a wrapper!

In [ ]:
# Let's first import the diagnostics
from mach3sbitools.diagnostics import SBCDiagnostic

# Now we can initialise the diagnoser
diag = SBCDiagnostic(simulator, inference_handler, Path("sbc_plots"))

In [ ]:
# First we need to sample from our prior
# Usually you'll want a few more samples than this but it'll do
diag.create_prior_samples(10000)

In [ ]:
# First we'll make the SBC rank plot, a good rank plot is basically flat! 
# More information can be found here: https://sbi.readthedocs.io/en/stable/how_to_guide/16_sbc.html
diag.rank_plot(1000, 20)

In [ ]:
# Next we'll get the expected coverage, this will tell you if you're
# over or under confident
# More information can be found here: https://sbi.readthedocs.io/en/stable/how_to_guide/15_expected_coverage.html

diag.expected_coverage(10000, 20)

In [ ]:
# Finally we'll run TARP, this assesses the accuracy of you predictor
# More information can be found here https://sbi.readthedocs.io/en/stable/how_to_guide/17_tarp.html
diag.tarp(10000)